<center>
    <img src="http://sct.inf.utfsm.cl/wp-content/uploads/2020/04/logo_di.png" style="width:60%">
    <h2> Taller de Computación Científica Aplicada</h2>
    <h2> [S]cientific [C]omputing [T]eam </a> </h2>
    <h2> Mayo 2026</h2>
</center>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github.com/tclaudioe/WorkshopAppliedSC/blob/main/Taller.ipynb)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tclaudioe/Scientific-Computing-V3/blob/main/Bonus%20-%20current/Bonus%20-%2000%20-%20Class%20zero.ipynb)

Correr el siguiente bloque solo 1 vez

In [1]:
##########################
# CoLab requirements
# https://stackoverflow.com/questions/44210656/how-to-check-if-a-module-is-installed-in-python-and-if-not-install-it-within-t
##########################

# https://pypi.org/project/colorama/

import importlib.util
import sys
import subprocess
import os
    
# install_colab_requirements 
libraries = ['numpy', 'scipy', 'matplotlib', 'colorama', 
            'bitstring', 'sympy', 'ipywidgets','pandas','pillow']

for library in libraries:
    # Check if the library is already installed
    if importlib.util.find_spec(library) is not None:
        print(f"{library} is already installed.")
    else:
        print(f"{library} is not installed. Installing...")
        # Install the library using pip
        subprocess.check_call([sys.executable, "-m", "pip", "install", library])
        print(f"{library} has been installed.")
# https://stackoverflow.com/questions/53581278/test-if-notebook-is-running-on-google-colab
if os.getenv("COLAB_RELEASE_TAG"):
    print('Installing LaTeX support in CoLab')
    # Adding LaTeX dependencies to CoLab: https://stackoverflow.com/questions/55746749/latex-equations-do-not-render-in-google-colaboratory-when-using-matplotlib
    !sudo apt install cm-super dvipng texlive-latex-extra texlive-latex-recommended
else:
   print('Running on local environment')

numpy is already installed.
scipy is already installed.
matplotlib is already installed.
colorama is already installed.
bitstring is already installed.
sympy is already installed.
ipywidgets is already installed.
pandas is already installed.
pillow is not installed. Installing...
pillow has been installed.
Running on local environment


In [2]:
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from numpy.linalg import svd
from scipy.ndimage import convolve1d
from scipy.linalg import toeplitz
from ipywidgets import widgets, interact, fixed, interactive, HBox, VBox, Layout # type: ignore

# Modelado de desenfoque en imágenes mediante operadores lineales

En este taller exploraremos el proceso de desenfoque de una imagen utilizando operadores lineales en forma de **matrices de convolución**, específicamente matrices Toeplitz construidas a partir de un núcleo Gaussiano.

El desenfoque de una imagen es una operación fundamental en procesamiento de imágenes y se puede modelar matemáticamente como una multiplicación matricial. 
En lugar de aplicar una función de convolución tradicional sobre una imagen, representaremos el desenfoque mediante una **operación lineal separable**, en la que una imagen bidimensional es multiplicada por matrices que difuminan por **filas** y por **columnas**.

Adicionalmente, se agregará **ruido** a la imagen desenfocada, con el objetivo de obtener una versión degradada de la imagen original.

El enfoque principal de este taller, y el desafío que se plantea, consiste en que ustedes logren **restaurar la imagen degradada** aplicando conceptos de álgebra lineal. 
Para ello, deberán utilizar **operaciones matriciales** junto con las propiedades de la **descomposición en valores singulares (SVD)**, con el fin de reconstruir una versión lo más **cercana** posible de la imagen original.

El procedimiento de desenfoque se modela con las matrices $A_c$ y $A_r$, las cuales operan sobre las columnas y filas de la imagen $X_{true}$, respectivamente. 
Es decir:
$$
\underbrace{A_c\,X_{\text{true}}\,A_r^\top}_{\displaystyle{B_{\text{true}}}} + \Xi = B_{\text{data}},
$$
donde:
- $X_{\text{true}}$: imagen original (en escala de grises).
- $A_c$ : matriz que aplica el desenfoque por columnas.
- $A_r$: matriz que aplica el desenfoque por filas.
- $\Xi$: ruido Gaussiano aditivo asociado al proceso de captura de la imagen.
- $B_{\text{true}}$: imagen desenfocada resultante sin considerar el ruido $\Xi$.
- $B_{\text{data}}$: imagen degradada que se quiere corregir. Esta es la data que se tiene a disposición.

Ambas matrices  $A_c$ y $A_r $ son construidas a partir de un filtro gaussiano discreto, y suelen ser matrices Toeplitz simétricas, y por simplicidad se usará $A_c=A_r$.

Este enfoque no solo permite aplicar desenfoques, sino también estudiar su inversión, lo que se utilizará más adelante para **restauración de imágenes** .

---

Funcion que retorna las 2 matrices de convolución $A_c$ y $A_r$:

In [3]:
# Construcción de operadores de 'desenfoque'
def blur_operator(m, n, sigmac = 0.01, sigmar = 0.01):
    """
    Descripción: Genera los operadores de convolución 1-D empleados
    ---------- 
    inputs:
    ----------
    m, n : int
        Dimensiones de la imagen (filas = m, columnas = n).
    sigmac, sigmar : float
        Desviaciones estándar para el desenfoque horizontal y vertical.
    ----------
    outputs:
    ----------
    Ac : ndarray (m, m)
        Matriz Toeplitz que difumina horizontalmente (columnas).
    Ar : ndarray (n, n)
        Matriz Toeplitz que difumina verticalmente (filas).
    """
    # --- 1. Nucleos 1-D ---------------------------------------------------
    c_pos = np.linspace(0, 1, m) # coordenadas normalizadas
    r_pos = np.linspace(0, 1, n)

    c = np.exp(-0.5 * (c_pos / sigmac) ** 2)
    r = np.exp(-0.5 * (r_pos / sigmar) ** 2)
    # umbral 1e-4 
    c[c < 1e-4] = 0.0
    r[r < 1e-4] = 0.0

    # Normalización: división por (2·sum − valor central)
    c /= (2 * c.sum() - c[0])
    r /= (2 * r.sum() - r[0])
    # --- 2. Matrices Toeplitz --------------------------------------------
    Ac = toeplitz(c)          # (m × m)
    Ar = toeplitz(r)          # (n × n)
    return Ac, Ar


Cargamos la imagen y la desenfocamos

In [4]:
def show_blurred_image(sigmac=0.01, sigmar=0.01):

    # Mostrar imagen original
    plt.figure(figsize=(12,4))

    ################################################################
    # Paso 1: Cargar imagen original y convertir a escala de grises
    img = Image.open("img.jpg").convert("L")
    img = img.resize((512, 512))
    X_true = np.array(img) / 255.0  # Imagen original normalizada

    plt.subplot(131)
    plt.imshow(img, cmap='gray')
    plt.axis('off')
    plt.title('Imagen original (X_true)')
    plt.tight_layout()

    ################################################################
    # Paso 2: Definir el operador de desenfoque
    Ac, Ar = blur_operator(512, 512, sigmac=sigmac, sigmar=sigmar)

    # Paso 3: Aplicar el operador de desenfoque
    B_blur = Ac @ X_true @ Ar.T

    # Mostrar imagen desenfocada
    plt.subplot(132)
    plt.imshow(B_blur, cmap='gray')
    plt.axis('off')
    plt.title('Imagen desenfocada (B_true)')
    plt.tight_layout()

    ################################################################
    # Paso 4: Cargar ruido definido previamente
    ruido=np.load('ruido.npy')  # Ruido gaussiano dado

    # Paso 5: Agregar ruido a la imagen desenfocada
    B_data = B_blur + ruido

    # Mostrar imagen desenfocada más ruido Gaussiano
    plt.subplot(133)
    plt.imshow(B_data, cmap='gray')
    plt.axis('off')
    plt.title('Imagen desenfocada + ruido (B_data)')
    plt.tight_layout()

    plt.show()

interact(show_blurred_image, sigmac=widgets.FloatSlider(min=0.001, max=0.1, step=0.001, value=0.01), 
         sigmar=widgets.FloatSlider(min=0.001, max=0.1, step=0.001, value=0.01))

interactive(children=(FloatSlider(value=0.01, description='sigmac', max=0.1, min=0.001, step=0.001), FloatSlid…

<function __main__.show_blurred_image(sigmac=0.01, sigmar=0.01)>

### Pasos para la reconstrucción

Para la realización de la reconstrucción de una **aproximación** la imagen $X_{\text{true}}$ desde la imagen $B_{\text{data}}$ se puede realizar _despejando_ $X_{\text{true}}$, es decir,
$$
\begin{align*}
    A_c\,X_{\text{true}}\,A_r^\top+\Xi &=B_{\text{data}},\tag{1.1}\\
    A_c\,X_{\text{true}}\,A_r^\top &=B_{\text{data}}-\Xi,\\
    X_{\text{true}}\,A_r^\top &=A_c^{-1}\left(B_{\text{data}}-\Xi\right),\\
    X_{\text{true}} &=A_c^{-1}\,\left(B_{\text{data}}-\Xi\right)\,A_r^{-\top},\\
    X_{\text{true}} &=\left(A_c^{-1}\,B_{\text{data}}-A_c^{-1}\,\Xi\right)\,A_r^{-\top},\\
    X_{\text{true}} &=A_c^{-1}\,B_{\text{data}}\,A_r^{-\top}-A_c^{-1}\,\Xi\,A_r^{-\top}.\tag{1.2}
\end{align*}$$
Esto nos indica que si obtenemos $A_c^{-1}\,B_{\text{data}}\,A_r^{-\top}$ directamente no obtendremos $X_{\text{true}}$ dado que desconocemos el término $\Xi$.
Incluso podemos determinar que la diferencia entre $X_{\text{true}}$ y $A_c^{-1}\,B_{\text{data}}\,A_r^{-\top}$ es exactamente $-A_c^{-1}\,\Xi\,A_r^{-\top}$, y si calculamos una norma matricial de la diferencia se obtiene,
$$
    \left\|X_{\text{true}}-A_c^{-1}\,B_{\text{data}}\,A_r^{-\top}\right\| = \left\|A_c^{-1}\,\Xi\,A_r^{-\top}\right\|.
$$
Entonces, para poder lograr reducir esta diferencia entre la imagen original $X_{\text{true}}$ y la imagen reconstruida se propone el siguiente procedimiento:
1. Descomponer el operador $A_c^{-1}$, es decir $A_c^{-1}=A_{c,\text{trunc}}^{-1}+A_{c,\text{res}}^{-1}$
2. Descomponer el operador $A_r^{-\top}$, es decir $A_r^{-\top}=A_{r,\text{trunc}}^{-\top}+A_{r,\text{res}}^{-\top}$

donde `trunc` denota el truncamiento del operador y `res` denota el residuo resultante del truncamiento.
El siguiente paso corresponde a reemplazar la descomposición de los operadores en el término $A_c^{-1}\,B_{\text{data}}\,A_r^{-\top}$ de la ecuación (1.2), es decir,
$$
\begin{align*}
    A_c^{-1}\,B_{\text{data}}\,A_r^{-\top} &= \left(A_{c,\text{trunc}}^{-1}+A_{c,\text{res}}^{-1}\right)
                                                B_{\text{data}}
                                                \left(A_{r,\text{trunc}}^{-\top}+A_{r,\text{res}}^{-\top}\right)\\
                                            &= A_{c,\text{trunc}}^{-1}\,B_{\text{data}}\,A_{r,\text{trunc}}^{-\top}+\Lambda_{\text{res}}, \tag{1.3}
\end{align*}
$$
donde 
$\Lambda_{\text{res}}=
A_{c,\text{trunc}}^{-1}\,B_{\text{data}}\,A_{r,\text{res}}^{-\top}
+A_{c,\text{res}}^{-1}\,B_{\text{data}}\,A_{r,\text{trunc}}^{-\top}
+A_{c,\text{res}}^{-1}\,B_{\text{data}}\,A_{r,\text{res}}^{-\top}$.
Notar que si no existe truncamiento, entonces la matriz $\Lambda_{\text{res}}$ sería la matriz nula.
Entonces, reemplazando la descomposición (1.3) en la ecuación (1.2) se obtiene,
$$
\begin{align*}
    X_{\text{true}} &= A_c^{-1}\,B_{\text{data}}\,A_r^{-\top}-A_c^{-1}\,\Xi\,A_r^{-\top}\\
                    &= \underbrace{A_{c,\text{trunc}}^{-1}\,B_{\text{data}}\,A_{r,\text{trunc}}^{-\top}}_{\displaystyle{X_{\text{recon}}}}+\Lambda_{\text{res}}-A_c^{-1}\,\Xi\,A_r^{-\top}.
\end{align*}
$$
Ahora, denotando $X_{\text{recon}}=A_{c,\text{trunc}}^{-1}\,B_{\text{data}}\,A_{r,\text{trunc}}^{-\top}$ como la imagen reconstruida, podemos obtener nuevamente la norma de la diferencia entre la imagen original y la imagen reconstruida de la siguiente forma,
$$
\begin{align*}
    \left\|X_{\text{true}}-X_{\text{recon}}\right\|=\left\|\Lambda_{\text{res}}-A_c^{-1}\,\Xi\,A_r^{-\top}\right\|.
\end{align*}
$$
Por lo tanto, si el truncamiento es exitoso se puede lograr que el error de la reconstrucción por medio de truncamiento sea menor que utilizar los operadores originales, es decir, estudiaremos las condiciones para que esto se logre lo siguiente,
$$
\left\|\Lambda_{\text{res}}-A_c^{-1}\,\Xi\,A_r^{-\top}\right\|=\left\|X_{\text{true}}-X_{\text{recon}}\right\|\leq\left\|X_{\text{true}}-A_c^{-1}\,B_{\text{data}}\,A_r^{-\top}\right\|= \left\|A_c^{-1}\,\Xi\,A_r^{-\top}\right\|.
$$
El éxito del procedimiento implica que se obtiene una imagen más cercana a la original por medio del truncamiento que utilizar los operadores sin truncar.
_Este procedimiento de aproximación es lo que se estudiará en este taller_.

**Su objetivo será implementar esta expresión paso a paso utilizando operaciones matriciales y aplicar la descomposición SVD para obtener la mejor estimación posible de la imagen original para la problemática particular que se le entregue.**

### Funciones que seran utilizadas


Funcion `np.linalg.svd(A, full_matrices=False)` --> obtiene la descomposicion en valores singulares (svd) de la matriz A.

In [5]:
A = np.array([[1., 2., 3.],
              [4., 5., 6.],
              [7., 8., 9.]])

Uc, Sc, VcT = np.linalg.svd(A, full_matrices=False)
print('Uc:',Uc)
print('Sc:',Sc)
print('VcT:',VcT)


Uc: [[-0.21483724  0.88723069  0.40824829]
 [-0.52058739  0.24964395 -0.81649658]
 [-0.82633754 -0.38794278  0.40824829]]
Sc: [1.68481034e+01 1.06836951e+00 3.33475287e-16]
VcT: [[-0.47967118 -0.57236779 -0.66506441]
 [-0.77669099 -0.07568647  0.62531805]
 [-0.40824829  0.81649658 -0.40824829]]


Funcion `np.argsort()` --> obtiene los i`dices originales ordenados de los mayores o menores elementos.

In [6]:
a = np.array([30, 10, 50, 20])

# Obtener los índices que ordenarían el array
indices_ordenados = np.argsort(a)

print("Índices ordenados menor a mayor:", indices_ordenados)  # [1 3 0 2]
print("Array ordenado:", a[indices_ordenados])  # [10 20 30 50]

indices_ordenados = np.argsort(a)[::-1]
print("Índices ordenados de mayor a menor:", indices_ordenados)  # [2 0 3 1]
print("Array ordenado:", a[indices_ordenados])  # [10 20 30 50]

n_indices_ordenados= np.argsort(a)[::-1][:2] #n=2
print("Los primeros n indices ordenados de mayor a menor:", n_indices_ordenados)  # [2 0]
print("Array ordenado:", a[n_indices_ordenados])  # [50 30]


Índices ordenados menor a mayor: [1 3 0 2]
Array ordenado: [10 20 30 50]
Índices ordenados de mayor a menor: [2 0 3 1]
Array ordenado: [50 30 20 10]
Los primeros n indices ordenados de mayor a menor: [2 0]
Array ordenado: [50 30]


Funcion `.ravel()` --> aplana un arreglo multidimensional, conviritendolo en un arreglo plano 1D, cada fila se concatena con la que sigue.

In [7]:
A = np.array([[1., 2., 3.],
              [4., 5., 6.],
              [7., 8., 9.]])

r = A.ravel()
print(r)

# funciona solo como vista, es decir, no se copia el array y A no se modifica, sigue siendo una matriz
print(A)

[1. 2. 3. 4. 5. 6. 7. 8. 9.]
[[1. 2. 3.]
 [4. 5. 6.]
 [7. 8. 9.]]


Funcion `np.zeros_like(A,dtype)` --> crea una matriz de ceros de la misma dimension que A.

In [8]:
copy=np.zeros_like(A)
print(copy)
#matriz booleana
copy=np.zeros_like(A, dtype=bool)
print(copy)


[[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]
[[False False False]
 [False False False]
 [False False False]]


Funciones utilizadas en conjunto


In [9]:
# Quiero marcar los primeros 3 elementos de la matriz mas grandes y dividirlo por 2

indices_ordenados = np.argsort(A.ravel())[::-1][:3]
print(A.ravel()[indices_ordenados], "\n")
B=A.ravel()[indices_ordenados] / 2
print(B,"\n")

# Dejar solo los 3 elementos que dividi por 2 en una matriz
Acopy = np.zeros_like(A)
Acopy.ravel()[indices_ordenados] = B
print("Matriz con los 3 elementos que dividi por 2")
print(Acopy,"\n")

# Dejar todos lo elementos mas los que dividi por 2
A.ravel()[indices_ordenados] = B
print("Matriz con los 3 elementos que dividi por 2")
print(A,"\n")    

[9. 8. 7.] 

[4.5 4.  3.5] 

Matriz con los 3 elementos que dividi por 2
[[0.  0.  0. ]
 [0.  0.  0. ]
 [3.5 4.  4.5]] 

Matriz con los 3 elementos que dividi por 2
[[1.  2.  3. ]
 [4.  5.  6. ]
 [3.5 4.  4.5]] 



# Funciones

Para la implementación de las preguntas, considere que **solo** tiene a su disposición las siguientes funciones:
* **np.linalg.svd(A, full_matrices=False)**: Calcula la descomposición en valores singulares (SVD) de la matriz **A**. Devuelve tres matrices: **U**, **Σ** (valores singulares) y **$V^\top$** tales que $A = U \Sigma V^\top$.
Por ejemplo `A = np.array([[1, 2], [3, 4], [5, 6]])`
le aplicamos la descomposicion svd `U, S, VT = np.linalg.svd(A, full_matrices=False)`
los output correspondientes serian:
    ```python
    U: [[-0.2298477   0.88346102]
        [-0.52474482  0.24078249]
        [-0.81964194 -0.40189603]]
    S: [9.52551809 0.51430058]
    VT: [[-0.61962948 -0.78489445]
         [-0.78489445  0.61962948]]

* **np.argsort(a)**: Devuelve los índices que ordenarían un array. Si el array es unidimensional, los índices corresponden al orden ascendente de los elementos. Para arrays multidimensionales, se aplica el ordenamiento aplanando el array primero.
`np.argsort([3, 1, 2])` devolvería `[1, 2, 0]`, ya que el índice del menor elemento osea el (1) es la posicion 1, seguido por el índice del siguiente menor elemento (2), que esta en la posicion 2, y finalmente el índice del mayor elemento (3), que esta en la posicion 0.
* **np.ravel(a)**: Devuelve una vista aplanada del array de entrada **a**. Si el array es multidimensional, se convierte en un array unidimensional sin copiar los datos siempre que sea posible.Por ejemplo, si se construye la matriz `a = np.array([[1,2],[3,4]])`, y se le aplica `np.ravel(a)` se obtiene `[1 2 3 4]`.

* **np.zeros_like(a)**: Devuelve un array de ceros con la misma forma y tipo que el array de entrada **a**. Esto es útil para inicializar matrices o arrays con valores nulos mientras se conserva la estructura del array original.  
Por ejemplo, si se construye la matriz `a = np.array([[1, 2], [3, 4]])`, y se le aplica `np.zeros_like(a)` se obtiene `array([[0, 0],[0, 0]])`

* **np.linalg.norm(x, ord='fro')**: Calcula la norma de Frobenius de una matriz **x**, que es la raíz cuadrada de la suma de los cuadrados de sus elementos. Es útil para medir la magnitud de una matriz o calcular errores relativos entre matrices.  
Por ejemplo, si se construye la matriz `x = np.array([[1, 2], [3, 4]])`, y se le aplica `np.linalg.norm(x, ord='fro')`, se obtiene el resultado `5.477225575051661`, ya que:  
$\sqrt{1^2 + 2^2 + 3^2 + 4^2} = \sqrt{30} \approx 5.477$.

* **np.diag(a)**: Esta función crea una matriz diagonal a partir de un vector plano (unidimensional). Los elementos del vector se colocan en la diagonal principal de una nueva matriz cuadrada, y el resto de los elementos se rellenan con ceros. Por ejemplo, si `a = np.array([1, 2, 3])`, entonces `np.diag(a)` produce la matriz:
    ```python
        [[1,0,0]
         [0,2,0]
         [0,0,3]]

## Proceso de truncamiento para limpiar el ruido y el desenfoque de la imagen $B_{\text{data}}$

Sea $A_c $ la matriz de convolución por columna, multiplicada por la izquierda a la imagen.

Construyendo la factorización SVD de $A_c$ obtenemos:
$$
A_c = U_c\,\Sigma_c\,V_c^\top 
$$
Para obtener la reconstrucción $X_{\text{recon}}$ de la imagen original $X_{\text{true}}$, se necesita construir $A_{c,\text{trunc}}^{-1}$, la cual se obtendrá por medio de la SVD anterior.
Es decir, definimos la reconstrucción de la siguiente forma 
$$
X_{\text{recon}} = A_{c,\text{trunc}}^{-1}\,B_{\text{data}}
$$
Utilizando la SVD de $A_c$, podemos obtener $A_{c,\text{trunc}}^{-1}$ de la siguiente forma,
$$
X_{\text{recon}} = V_c\,\widehat{\Sigma}_c\,U_c^\top\,B_{\text{data}}.
$$
donde,
$$
\left(\widehat{\Sigma}_c\right)_{i,j} =
\begin{cases}
    \frac{1}{(\sigma_c)_i} &  \text{ si $(\sigma_c)_i> \alpha$ y $i=j$}, \\
    0 & \text{en todo otro caso},
\end{cases}
$$
donde $i,j\in \{1,2,3,\dots,n\}$, es decir se indexan desde $1$.
**Esto significa que solo seleccionaremos los valores singulares $\sigma_c$ mayores a $\alpha$ para poder truncar el operador.**
Así, $\widehat{\Sigma}_c$ solo conserva la información de los valores singulares significativos, eliminando el efecto de los valores singulares pequeños asociados principalmente al ruido.
Ahora, por conveniencia, se define la siguiente matriz,
$$
\widehat{B} = U_c^\top\,B_{\text{data}}
$$
Así, la reconstrucción queda:
$$
X_{\text{recon}} = V_c \left( \widehat{\Sigma}_c\,\widehat{B} \right)
$$

# Implementación

Este código construye la matriz $\widehat{\Sigma}_c^{-1}$ definida anteriormente considerando el truncamiento $\alpha$.
Se recibe como parámetros un vector con los valores singulares de $A_c$ y el umbral $\alpha$.

In [10]:
def matriz_Sigma_hat(sigma,alpha):
    """
    input:
    sigma : array(n)
        Vector sigma (contiene los valores sigulares).    
    alpha : float
        Tolerancia para seleccionar los valores singulares.              
    output:
    Sigma_hat: ndarray (n, n)
        Devuelve una matriz con el reciproco de los valores singulares mayores a alpha.
    """
    flat_sigma_trunc = np.zeros_like(sigma)
    flat_sigma_trunc[sigma > alpha] = 1 / sigma[sigma > alpha]
    sigma_trunc = flat_sigma_trunc
    Sigma_hat=np.diag(sigma_trunc)
    
    return Sigma_hat

Implementación de la reconstrucción

In [11]:
def restaurar_imagen_c(B_data,alpha,sigmac):
    """
    input:  
    B_data : ndarray (m, n)
        Imagen borrosa y ruidosa (con ruido gaussiano).
    alpha : float
        Tolerancia para seleccionar los valores singulares.
    sigmac : float
        Parámetro para la función blur_operator.
    output:
    X_recon : ndarray (m, n)
        Imagen reconstruida (con ruido y desenfoque).
    """
    # Matriz Toeplitz que difumina horizontalmente (columnas).
    Ac, _  = blur_operator(512, 512, sigmac=sigmac)

    Uc, Sc, VcT = np.linalg.svd(Ac, full_matrices=False)
    Vc = VcT.T
    B_hat = Uc.T @ B_data
    Sigma_hat = matriz_Sigma_hat(Sc,alpha)
    X_recon = Vc @ (Sigma_hat @ B_hat)
    
    return X_recon, Sc 

Implementación para desenfocar imagen.

In [12]:
def blur_image_c(sigmac=0.01, sigmar=0.01):

    ################################################################
    # Paso 1: Cargar imagen original y convertir a escala de grises
    img = Image.open("img.jpg").convert("L")
    img = img.resize((512, 512))
    X_true = np.array(img) / 255.0  # Imagen original normalizada

    ################################################################
    # Paso 2: Definir el operador de desenfoque
    Ac, Ar = blur_operator(512, 512, sigmac=sigmac, sigmar=sigmar)

    # Paso 3: Aplicar el operador de desenfoque
    # B_blur = Ac @ X_true @ Ar.T
    B_blur = Ac @ X_true

    ################################################################
    # Paso 4: Cargar ruido definido previamente
    ruido=np.load('ruido.npy')  # Ruido gaussiano dado

    # Paso 5: Agregar ruido a la imagen desenfocada
    B_data = B_blur + ruido
    
    # Retornar imagen desenfocada con ruido Gaussiano    
    return B_data

# Experimento numérico para recuperar imagen

In [ ]:
B_data = blur_image_c(sigmac=0.01)

def restaurar_variando_alpha_y_sigmac(B_data,alpha,sigmac):
    """
    input:  
    B_data : ndarray (m, n)
        Imagen borrosa y ruidosa (con ruido gaussiano).
    alpha : float
        Tolerancia para seleccionar los valores singulares.
    sigmac : float
        Parámetro para la función blur_operator.
    output:
    X_recon : ndarray (m, n)
        Imagen reconstruida (con ruido y desenfoque).
    """
    X_recon, Sc = restaurar_imagen_c(B_data,alpha,sigmac)
    
    plt.figure(figsize=(12,4))
    
    plt.subplot(131)
    plt.imshow(B_data, cmap='gray')
    plt.title('Imagen Borrosa y Ruidosa')
    
    plt.subplot(132)
    plt.semilogy(Sc, '.')
    plt.title('Valores Singulares')
    plt.plot([0, len(Sc)], [alpha, alpha], 'r--', label=f'alpha={alpha}')
    plt.grid(True)
    
    plt.subplot(133)
    plt.imshow(X_recon, cmap='gray')
    plt.title('Imagen Reconstruida')
    plt.show()
    
    # return X_recon

interact(restaurar_variando_alpha_y_sigmac, B_data=fixed(B_data),
         alpha=widgets.FloatLogSlider(min=-20, max=2, step=0.1, value=2), 
         sigmac=widgets.FloatLogSlider(min=-5, max=1, step=0.01, value=0.01))

interactive(children=(FloatLogSlider(value=2.0, description='alpha', max=2.0, min=-20.0), FloatLogSlider(value…

<function __main__.restaurar_variando_alpha_y_sigmac(B_data, alpha, sigmac)>